# 2CDE: within-burst dynamics without a kinetic model

A burst-averaged FRET efficiency cannot tell a molecule that sat at
$E = 0.5$ from one that switched between 0.2 and 0.8 during the transit.
The **two-channel kernel density estimator** (2CDE, Tomov et al., Biophys. J.
2012, [doi:10.1016/j.bpj.2011.11.4025](https://doi.org/10.1016/j.bpj.2011.11.4025)) is a per-burst feature that
does: around every photon a kernel density of the donor and of the acceptor
stream is evaluated (a Laplace or Gaussian kernel of width $\tau$,
typically 50–100 µs), and

\begin{align}\mathrm{FRET\text{-}2CDE} = 110 - 100\,[(E)_D + (1-E)_A]\end{align}

compares the local FRET efficiency seen from donor photons with that seen from
acceptor photons. A static burst gives ≈ 10 whichever kernel; a burst whose
FRET changes on the millisecond scale gives 30–100, because donor photons
cluster where E is low and acceptor photons where it is high. ALEX-2CDE does
the same for donor- vs acceptor-excitation streams and flags acceptor blinking.

`tttrlib.TwoCDE` computes the KDEs on the *whole* photon stream (no burst-edge
artefacts) and reduces them per burst; it is a bit-exact port of FRETBursts'
``kde_laplace`` / ``kde_gaussian`` and 4× faster. This example simulates static
and dynamic bursts and shows the dynamic ones lift off the static line.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tttrlib

rng = np.random.default_rng(5)
RES = 1e-7                       # seconds per macro-time tick

## Simulate bursts as photon streams
Donor = channel 0, acceptor = channel 1. Each burst is a Poisson stream at
100 kHz for ~1.5 ms; static bursts keep one E, dynamic bursts switch between
two E values every ~400 us. Bursts are separated by 10 ms of nothing.



In [ ]:
def burst_photons(kind, e_static=0.5, e_pair=(0.2, 0.8), rate=1e5, duration=1.5e-3, switch=4e-4):
    n = rng.poisson(rate * duration)
    t = np.sort(rng.uniform(0, duration, n))
    if kind == "static":
        e = np.full(n, e_static)
    else:
        seg = (t // switch).astype(int) % 2
        e = np.where(seg == 0, e_pair[0], e_pair[1])
    ch = (rng.random(n) < e).astype(np.int8)          # 1 = acceptor
    return t, ch


kinds = ["static"] * 60 + ["dynamic"] * 60
macro, chan, bounds = [], [], []
t_offset = 0.0
for kind in kinds:
    t, ch = burst_photons(kind)
    start = len(macro)
    macro.extend(((t + t_offset) / RES).astype(np.int64))
    chan.extend(ch)
    bounds.append((start, len(macro) - 1))             # inclusive photon index range
    t_offset += 1.5e-3 + 10e-3
macro = np.asarray(macro, dtype=np.uint64)
chan = np.asarray(chan, dtype=np.int8)
bounds = np.asarray(bounds, dtype=np.int64)
tttr = tttrlib.TTTR(macro, np.zeros(macro.size, np.uint16), chan, np.zeros(macro.size, np.int8))
tttr.header.set_macro_time_resolution(RES)
print(f"{len(kinds)} bursts, {macro.size} photons")

## FRET-2CDE with both kernels



In [ ]:
two = tttrlib.TwoCDE(tttr)
two.set_donor([0])
two.set_acceptor([1])
tau = 100e-6                                           # kernel width in seconds
two.compute(bounds, tau, tttrlib.TwoCDE.FRET_2CDE, tttrlib.TwoCDE.LAPLACE)
cde_laplace = np.asarray(two.get_two_cde()).copy()
two.compute(bounds, tau, tttrlib.TwoCDE.FRET_2CDE, tttrlib.TwoCDE.GAUSSIAN)
cde_gauss = np.asarray(two.get_two_cde()).copy()

# burst-averaged E for the x axis
E = np.array([chan[s:e + 1].mean() for s, e in bounds])
is_dyn = np.array([k == "dynamic" for k in kinds])

for name, v in (("Laplace", cde_laplace), ("Gaussian", cde_gauss)):
    print(f"FRET-2CDE ({name}): static {np.nanmean(v[~is_dyn]):5.1f} +- {np.nanstd(v[~is_dyn]):4.1f}, "
          f"dynamic {np.nanmean(v[is_dyn]):5.1f} +- {np.nanstd(v[is_dyn]):4.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, name, v in zip(axes, ("Laplace kernel", "Gaussian kernel"), (cde_laplace, cde_gauss)):
    ax.scatter(E[~is_dyn], v[~is_dyn], s=14, label="static E = 0.5")
    ax.scatter(E[is_dyn], v[is_dyn], s=14, label="dynamic 0.2 <-> 0.8")
    ax.axhline(10, color="0.5", ls="--", lw=1)
    ax.set_xlabel("burst-averaged E")
    ax.set_title(f"FRET-2CDE, {name}, tau = {tau * 1e6:.0f} us")
    ax.legend()
axes[0].set_ylabel("FRET-2CDE")
fig.tight_layout()

## ALEX-2CDE: acceptor blinking under alternating excitation
With ALEX/PIE, donor-excitation photons (DexDem + DexAem) and
acceptor-excitation photons (AexAem) are two more streams; a burst in which
the acceptor blinks off for part of the transit gives ALEX-2CDE far from 100.
Simulate the streams as separate channels: 0/1 under donor excitation,
2 = acceptor under acceptor excitation, with half the bursts blinking.



In [ ]:
macro2, chan2, bounds2, blink = [], [], [], []
t_offset = 0.0
for i in range(120):
    n_dex = rng.poisson(120); n_aex = rng.poisson(120)
    t_d = np.sort(rng.uniform(0, 1.5e-3, n_dex))
    t_a = np.sort(rng.uniform(0, 1.5e-3, n_aex))
    if i % 2:                                          # acceptor dark in the second half
        t_a = t_a[t_a < 0.75e-3]
    ch_d = (rng.random(t_d.size) < 0.5).astype(np.int8)
    tt = np.concatenate([t_d, t_a]); cc = np.concatenate([ch_d, np.full(t_a.size, 2, np.int8)])
    order = np.argsort(tt)
    start = len(macro2)
    macro2.extend(((tt[order] + t_offset) / RES).astype(np.int64)); chan2.extend(cc[order])
    bounds2.append((start, len(macro2) - 1)); blink.append(bool(i % 2))
    t_offset += 1.5e-3 + 10e-3
macro2 = np.asarray(macro2, np.uint64); chan2 = np.asarray(chan2, np.int8)
bounds2 = np.asarray(bounds2, np.int64); blink = np.asarray(blink)
tttr2 = tttrlib.TTTR(macro2, np.zeros(macro2.size, np.uint16), chan2, np.zeros(macro2.size, np.int8))
tttr2.header.set_macro_time_resolution(RES)
alex = tttrlib.TwoCDE(tttr2)
alex.set_donor_excitation([0, 1])
alex.set_acceptor_excitation([2])
alex.compute(bounds2, tau, tttrlib.TwoCDE.ALEX_2CDE, tttrlib.TwoCDE.LAPLACE)
alex_cde = np.asarray(alex.get_two_cde())
print(f"ALEX-2CDE: steady {np.nanmean(alex_cde[~blink]):.1f}, blinking {np.nanmean(alex_cde[blink]):.1f}")

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.hist(alex_cde[~blink], bins=30, alpha=0.7, label="steady acceptor")
ax.hist(alex_cde[blink], bins=30, alpha=0.7, label="acceptor blinks")
ax.set_xlabel("ALEX-2CDE")
ax.set_ylabel("bursts")
ax.legend()
fig.tight_layout()
plt.show()